# Indexing and Selecing data

The axis labeling information in pandas objects serves many purposes:

* Identifies data (i.e. provides metadata) using known indicators, important for analysis, visualization, and interactive console display.
* Enables automatic and explicit data alignment.
* Allows intuitive getting and setting of subsets of the data set.

In this section, we will focus on the final point: namely, how to slice, dice, and generally get and set subsets of pandas objects. The primary focus will be on Series and DataFrame as they have received more development attention in this area.


### Different choices for indexing

Object selection has had a number of user-requested additions in order to support more explicit location based indexing. pandas now supports three types of multi-axis indexing.

* `.loc` is primarily label based, but may also be used with a boolean array. `.loc` will raise `KeyError` when the items are not found. Allowed inputs are:

  > * A single label, e.g. `5` or `'a'` (Note that `5` is interpreted as a *label* of the index. This use is **not** an integer position along the index.).
  > * A list or array of labels `['a', 'b', 'c']`.
  > * A slice object with labels `'a':'f'` (Note that contrary to usual Python slices, **both** the start and the stop are included, when present in the index!
  > * A boolean array (any `NA` values will be treated as `False`).
  > * A `callable` function with one argument (the calling Series or DataFrame) and that returns valid output for indexing (one of the above).
  > * A tuple of row (and column) indices whose elements are one of the above inputs.
  >
  

* `.iloc` is primarily integer position based (from `0` to `length-1` of the axis), but may also be used with a boolean array. `.iloc` will raise `IndexError` if a requested indexer is out-of-bounds, except *slice* indexers which allow out-of-bounds indexing. (this conforms with Python/NumPy *slice* semantics). Allowed inputs are:

  > * An integer e.g. `5`.
  > * A list or array of integers `[4, 3, 0]`.
  > * A slice object with ints `1:7`.
  > * A boolean array (any `NA` values will be treated as `False`).
  > * A `callable` function with one argument (the calling Series or DataFrame) and that returns valid output for indexing (one of the above).
  > * A tuple of row (and column) indices whose elements are one of the above inputs.
  >

* `.loc`, `.iloc`, and also `[]` indexing can accept a `callable` as indexer. 

In [1]:
import numpy as np
import pandas as pd

ser = pd.Series(range(5), index= list("abcde"))

In [2]:
ser

a    0
b    1
c    2
d    3
e    4
dtype: int64

In [3]:
ser.loc[["a", "c", "e"]]

a    0
c    2
e    4
dtype: int64

In [4]:
df = pd.DataFrame(np.arange(25).reshape(5, 5), index = list("abcde"), columns = list("abcde"))

In [5]:
df

,a,b,c,d,e
a,0,1,2,3,4
b,5,6,7,8,9
c,10,11,12,13,14
d,15,16,17,18,19
e,20,21,22,23,24


In [6]:
df.loc[["a" , "c", "e"], ["b", "d"]]

,b,d
a,1,3
c,11,13
e,21,23


### Basics
As mentioned when introducing the data structures in the previous section, the primary function of indexing with `[]` (a.k.a. `__getitem__` for those familiar with implementing class behavior in Python) is selecting out lower-dimensional slices. The following table shows return type values when indexing pandas objects with `[]`:

| Object Type | Selection        | Return Value Type                 |
| ----------- | ---------------- | --------------------------------- |
| Series      | `series[label]`  | scalar value                      |
| DataFrame   | `frame[colname]` | `Series` corresponding to colname |



In [7]:
# creating a simple time series data set to use for illustrating the indexing functionality:
dates = pd.date_range('1/1/2000', periods = 8)
df = pd.DataFrame(np.random.randn(8, 4),
                  index=dates, columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
2000-01-01,0.381918,-0.178866,-0.280091,-2.173926
2000-01-02,-2.165132,0.095747,0.599248,0.348885
2000-01-03,1.412580,0.028573,-0.702325,0.981729
2000-01-04,-0.492163,-0.565135,-0.502933,1.999355
2000-01-05,-0.293873,0.697005,0.694528,0.413838
2000-01-06,0.821495,0.085911,-0.790345,-0.040348
2000-01-07,0.326280,-1.481917,-0.470283,-0.741031
2000-01-08,-1.550474,0.015126,2.158909,3.557245


In [8]:
# thus as per above we have the most basic indexing using [].
s = df["A"]
s[dates[5]]

0.8214946817811405

In [9]:
# You can pass a list of columns to [] to select columns in that order. If a column is not contained in the DataFrame, an exception will be raised. Multiple columns can also be set in this manner:
df[['B', 'A']] = df[['A','B']]
df

,A,B,C,D
2000-01-01,-0.178866,0.381918,-0.280091,-2.173926
2000-01-02,0.095747,-2.165132,0.599248,0.348885
2000-01-03,0.028573,1.412580,-0.702325,0.981729
2000-01-04,-0.565135,-0.492163,-0.502933,1.999355
2000-01-05,0.697005,-0.293873,0.694528,0.413838
2000-01-06,0.085911,0.821495,-0.790345,-0.040348
2000-01-07,-1.481917,0.326280,-0.470283,-0.741031
2000-01-08,0.015126,-1.550474,2.158909,3.557245


In [10]:
df

,A,B,C,D
2000-01-01,-0.178866,0.381918,-0.280091,-2.173926
2000-01-02,0.095747,-2.165132,0.599248,0.348885
2000-01-03,0.028573,1.412580,-0.702325,0.981729
2000-01-04,-0.565135,-0.492163,-0.502933,1.999355
2000-01-05,0.697005,-0.293873,0.694528,0.413838
2000-01-06,0.085911,0.821495,-0.790345,-0.040348
2000-01-07,-1.481917,0.326280,-0.470283,-0.741031
2000-01-08,0.015126,-1.550474,2.158909,3.557245


In [11]:
# pandas aligns all AXES when setting Series and DataFrame from .loc.
#this will not modify df because the column alignment is before value assignment.

In [12]:
df[['A', 'B']]

,A,B
2000-01-01,-0.178866,0.381918
2000-01-02,0.095747,-2.165132
2000-01-03,0.028573,1.412580
2000-01-04,-0.565135,-0.492163
2000-01-05,0.697005,-0.293873
2000-01-06,0.085911,0.821495
2000-01-07,-1.481917,0.326280
2000-01-08,0.015126,-1.550474


In [13]:
df.loc[:, ['B', 'A']] = df[['A', 'B']]

In [14]:
df[['A', 'B']]

,A,B
2000-01-01,-0.178866,0.381918
2000-01-02,0.095747,-2.165132
2000-01-03,0.028573,1.412580
2000-01-04,-0.565135,-0.492163
2000-01-05,0.697005,-0.293873
2000-01-06,0.085911,0.821495
2000-01-07,-1.481917,0.326280
2000-01-08,0.015126,-1.550474


In [15]:
# The correct way to swap column values is by using raw values:
df.loc[:, ['B', 'A']] = df[['A', 'B']].to_numpy()

In [16]:
df[["A", "B"]]

,A,B
2000-01-01,0.381918,-0.178866
2000-01-02,-2.165132,0.095747
2000-01-03,1.412580,0.028573
2000-01-04,-0.492163,-0.565135
2000-01-05,-0.293873,0.697005
2000-01-06,0.821495,0.085911
2000-01-07,0.326280,-1.481917
2000-01-08,-1.550474,0.015126


### Selection by label

In [17]:
# .loc is strict when you present slicers that are not compatible (or convertible) with the index type. For example using integers in a DatetimeIndex. These will raise a TypeError.

dfl = pd.DataFrame(np.random.randn(5,4),
                  columns = list("ABCD"),
                  index = pd.date_range('20130101', periods = 5))

dfl

,A,B,C,D
2013-01-01,0.239948,-0.796730,-1.470406,-1.089378
2013-01-02,1.540372,-0.882989,-0.508140,0.420517
2013-01-03,0.189356,-1.563110,0.638640,1.098991
2013-01-04,-0.227355,1.508911,0.517457,-0.089565
2013-01-05,-0.481311,-0.612328,-0.125061,1.617700


In [18]:
dfl.loc[2:3]

TypeError: cannot do slice indexing on DatetimeIndex with these indexers [2] of type int

In [19]:
# String likes in slicing can be convertible to the type of the index and lead to natural slicing.
dfl.loc['20130102':'20130104']

,A,B,C,D
2013-01-02,1.540372,-0.882989,-0.508140,0.420517
2013-01-03,0.189356,-1.563110,0.638640,1.098991
2013-01-04,-0.227355,1.508911,0.517457,-0.089565


Pandas provides a suite of methods in order to have **purely label based indexing**. This is a strict inclusion based protocol. Every label asked for must be in the index, or a `KeyError` will be raised. When slicing, both the start bound **AND** the stop bound are *included*, if present in the index. Integers are valid labels, but they refer to the label **and not the position**.

The `.loc` attribute is the primary access method. The following are valid inputs:

* A single label, e.g. `5` or `'a'` (Note that `5` is interpreted as a *label* of the index. This use is **not** an integer position along the index.).
* A list or array of labels `['a', 'b', 'c']`.
* A slice object with labels `'a':'f'`. Note that contrary to usual Python slices, **both** the start and the stop are included, when present in the index!
* A boolean array.
* A `callable`.


In [20]:
s1 = pd.Series(np.random.randn(6), index = list('abcdef'))
s1

a   -0.012052
b   -2.287153
c    0.599532
d    0.295180
e    0.116901
f   -0.179015
dtype: float64

In [21]:
s1.loc['c':]

c    0.599532
d    0.295180
e    0.116901
f   -0.179015
dtype: float64

In [22]:
s1.loc['b']

-2.287153431936164

In [23]:
s1.loc["c":] = 0
s1

a   -0.012052
b   -2.287153
c    0.000000
d    0.000000
e    0.000000
f    0.000000
dtype: float64

In [24]:
# With a dataframe
df1 = pd.DataFrame(np.random.randn(6, 4),
                   index=list('abcdef'),
                   columns=list('ABCD'))
df1

,A,B,C,D
a,-0.798031,-0.304025,0.078322,-0.316631
b,-1.997695,-1.576126,-1.435220,-0.450541
c,-2.382290,-1.171037,-0.653851,-1.534870
d,0.443017,0.976378,0.821755,-1.778184
e,1.641096,1.303303,-0.074420,0.006712
f,-2.562242,0.360371,0.449569,-1.052435


In [25]:
# Accessing via label slices:
df1.loc['d':, 'A':'C']

,A,B,C
d,0.443017,0.976378,0.821755
e,1.641096,1.303303,-0.074420
f,-2.562242,0.360371,0.449569


In [26]:
# For getting a cross section using a label (equivalent to df.xs('a')):
df1.loc["a"]

A   -0.798031
B   -0.304025
C    0.078322
D   -0.316631
Name: a, dtype: float64

In [27]:
# For getting values with a boolean array:
df1.loc['a'] > 0

A    False
B    False
C     True
D    False
Name: a, dtype: bool

In [28]:
df1.loc[:, df1.loc['a'] > 0]

,C
a,0.078322
b,-1.435220
c,-0.653851
d,0.821755
e,-0.074420
f,0.449569


In [29]:
# NA Values in a boolean array propogate as False:
mask = pd.array([True, False, True, False, pd.NA, False], dtype="boolean")
mask

<BooleanArray>
[True, False, True, False, <NA>, False]
Length: 6, dtype: boolean

In [30]:
df1[mask]

,A,B,C,D
a,-0.798031,-0.304025,0.078322,-0.316631
c,-2.382290,-1.171037,-0.653851,-1.534870


The mask says:

```text
True
False
True
False
NA
False
```

Pandas interprets:

```text
True  → keep
False → remove
NA    → treat as False
```

So conceptually:

```text
a → True  → KEEP
b → False → REMOVE
c → True  → KEEP
d → False → REMOVE
```


In [31]:
# for getting a value explicitly:
# this is also equivalent to ``df1.at['a', 'A']``
df1.loc['a', 'A']

-0.7980310013805667

### Slicing with labels

In [32]:
# When using .loc with slices, if both the start and the stop labels are present in the index, then elements located between the two (including them) are returned:

s = pd.Series(list('abcde'), index = [0, 3, 2, 5, 4])
s.loc[3:5]

3    b
2    c
5    d
dtype: object

In [33]:
# If the index is sorted, and can be compared against start and stop labels, then slicing will still work as expected, by selecting labels which rank between the two:
s.sort_index()

0    a
2    c
3    b
4    e
5    d
dtype: object

In [34]:
s.sort_index().loc[1:6]

2    c
3    b
4    e
5    d
dtype: object

However, if at least one of the two is absent and the index is not sorted, an error will be raised (since doing otherwise would be computationally expensive, as well as potentially ambiguous for mixed type indexes). For instance, in the above example, `s.loc[1:6]` would raise `KeyError`.


In [35]:
s = pd.Series(list('abcdef'), index=[0, 3, 2, 5, 4, 2])
s.loc[3:5]

3    b
2    c
5    d
dtype: object

Also, if the index has duplicate labels *and* either the start or the stop label is duplicated, an error will be raised. For instance, in the above example, `s.loc[2:5]` would raise a `KeyError`.

When using a slice with a step, such as `.loc[start:stop:step]`, note that *start* and *stop* are interpreted as **labels**, while *step* is applied over the **positional index** within that label range. This means a stepped slice will behave differently than using the labels `range(start, stop, step)` when the index is not contiguous integers.


In [36]:
# For example, in a Series with a non-contiguous integer index:
s = pd.Series(range(10), index=[0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
s

0     0
5     1
10    2
15    3
20    4
25    5
30    6
35    7
40    8
45    9
dtype: int64

In [37]:
s.loc[10:50:5] # (10), then skip 3 positions → 35 only

10    2
35    7
dtype: int64

The first applies step across positional locations between the start/stop labels. The second selects each label directly.

Similarly, with a string-based index, the behavior is identical:

In [38]:
s = pd.Series(range(10), index=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j'])
s

a    0
b    1
c    2
d    3
e    4
f    5
g    6
h    7
i    8
j    9
dtype: int64

In [39]:
s.loc['b':'i':2]# Start at 'b' (position 1), stop at 'i' (position 8), step 2 positions → 'b', 'd', 'f', 'h'

b    1
d    3
f    5
h    7
dtype: int64

In [40]:
s.loc[['b', 'd', 'f', 'h']] # explicit label selection
# In both cases, start and stop determine the label boundaries (inclusive), while step skips positions within that range, regardless of the index type.

b    1
d    3
f    5
h    7
dtype: int64

### Selection by position

pandas provides a suite of methods in order to get **purely integer based indexing**. The semantics follow closely Python and NumPy slicing. These are `0-based` indexing. When slicing, the start bound is *included*, while the upper bound is *excluded*. Trying to use a non-integer, even a **valid** label will raise an `IndexError`.

The `.iloc` attribute is the primary access method. The following are valid inputs:

* An integer e.g. `5`.
* A list or array of integers `[4, 3, 0]`.
* A slice object with ints `1:7`.
* A boolean array.
* A `callable`.
* A tuple of row (and column) indexes, whose elements are one of the above types.


In [41]:
s1 = pd.Series(np.random.randn(5), index=list(range(0, 10, 2)))
s1

0   -1.164988
2    1.609629
4    0.822479
6   -0.066376
8   -0.481221
dtype: float64

In [42]:
s1.iloc[:3]

0   -1.164988
2    1.609629
4    0.822479
dtype: float64

In [43]:
s1.iloc[3]

-0.06637557948436511

In [44]:
# Note that setting works as well:
s1.iloc[:3] = 0
s1

0    0.000000
2    0.000000
4    0.000000
6   -0.066376
8   -0.481221
dtype: float64

In [45]:
# with a dataframe:

In [46]:
df1 = pd.DataFrame(np.random.randn(6,4),
                  index = list(range(0, 12, 2)),
                  columns= list(range(0, 8, 2)))
df1

,0,2,4,6
0,0.724739,1.019287,-0.150252,-1.442600
2,0.535257,-0.294666,0.609707,0.525795
4,-0.003624,-0.289934,0.830572,1.303991
6,-2.298959,1.048796,0.867630,-0.869239
8,-1.986691,-1.035160,-2.846162,-0.180748
10,2.150935,0.628694,0.704604,1.186429


In [47]:
# select via integer slicing
df1.iloc[:3]

,0,2,4,6
0,0.724739,1.019287,-0.150252,-1.442600
2,0.535257,-0.294666,0.609707,0.525795
4,-0.003624,-0.289934,0.830572,1.303991


In [48]:
# select via integer list:
df1.iloc[[1, 3, 5], [1, 3]]

,2,6
2,-0.294666,0.525795
6,1.048796,-0.869239
10,0.628694,1.186429


In [49]:
df1.iloc[1:3, :]

,0,2,4,6
2,0.535257,-0.294666,0.609707,0.525795
4,-0.003624,-0.289934,0.830572,1.303991


In [50]:
df1.iloc[:, 1:3]

,2,4
0,1.019287,-0.150252
2,-0.294666,0.609707
4,-0.289934,0.830572
6,1.048796,0.867630
8,-1.035160,-2.846162
10,0.628694,0.704604


In [51]:
# this is also equivalent to ``df1.iat[1,1]``
df1.iloc[1,1]

-0.2946657236378808

In [52]:
# For getting a cross section using an integer position (equiv to df.xs(1)):
df1.iloc[1]

0    0.535257
2   -0.294666
4    0.609707
6    0.525795
Name: 2, dtype: float64

In [53]:
# Out of range slice indexes are handled gracefully just as in Python/NumPy.
x = list('abcdef')
x

['a', 'b', 'c', 'd', 'e', 'f']

In [54]:
x[4:10]

['e', 'f']

In [55]:
x[8:10]

[]

In [56]:
s = pd.Series(x)
s

0    a
1    b
2    c
3    d
4    e
5    f
dtype: object

In [57]:
s.iloc[4:10]

4    e
5    f
dtype: object

In [58]:
s.iloc[8:10]

Series([], dtype: object)

In [59]:
# Note that using slices that go out of bounds can result in an empty axis (e.g. an empty DataFrame being returned).
dfl = pd.DataFrame(np.random.randn(5,2), columns = list('AB'))
dfl

,A,B
0,0.604060,-0.186586
1,0.240344,-0.231141
2,-0.830976,0.764352
3,-0.581334,-0.121671
4,-1.841683,0.338119


In [60]:
dfl.iloc[:, 2:3]

""
0
1
2
3
4


In [61]:
dfl.iloc[:, 1:3]

,B
0,-0.186586
1,-0.231141
2,0.764352
3,-0.121671
4,0.338119


In [62]:
dfl.iloc[4:6]

,A,B
4,-1.841683,0.338119


In [63]:
# A single indexer that is out of bounds will raise an IndexError. A list of indexers where any element is out of bounds will raise an IndexError.
dfl.iloc[[4, 5, 6]]

IndexError: positional indexers are out-of-bounds

In [64]:
dfl.iloc[:, 4]

IndexError: single positional indexer is out-of-bounds

### Selection by callable
`.loc`, `.iloc`, and also `[]` indexing can accept a callable as indexer. The callable must be a function with one argument (the calling Series or DataFrame) that returns valid output for indexing

In [65]:
df1 = pd.DataFrame(np.random.randn(6,4),
                  index = list('abcdef'),
                  columns = list('ABCD'))
df1

,A,B,C,D
a,1.970393,-1.359079,-0.293214,-0.297579
b,0.204419,-0.448067,0.414499,0.053488
c,0.939789,2.399218,1.060424,-1.089277
d,0.351338,-0.368559,-0.829085,0.738131
e,0.577461,0.074074,-0.402376,1.372572
f,-1.174814,0.367682,1.279992,-1.979070


In [66]:
df1.loc[lambda df:df['A'] > 0, :]

,A,B,C,D
a,1.970393,-1.359079,-0.293214,-0.297579
b,0.204419,-0.448067,0.414499,0.053488
c,0.939789,2.399218,1.060424,-1.089277
d,0.351338,-0.368559,-0.829085,0.738131
e,0.577461,0.074074,-0.402376,1.372572


In [67]:
df1.loc[:, lambda df: ['A', 'B']]

,A,B
a,1.970393,-1.359079
b,0.204419,-0.448067
c,0.939789,2.399218
d,0.351338,-0.368559
e,0.577461,0.074074
f,-1.174814,0.367682


In [68]:
df1.iloc[:, lambda df: [0,1]]

,A,B
a,1.970393,-1.359079
b,0.204419,-0.448067
c,0.939789,2.399218
d,0.351338,-0.368559
e,0.577461,0.074074
f,-1.174814,0.367682


In [69]:
df1[lambda df: df.columns[0]]

a    1.970393
b    0.204419
c    0.939789
d    0.351338
e    0.577461
f   -1.174814
Name: A, dtype: float64

In [70]:
# you can use callable indexing in Series
df1['A'].loc[lambda s: s > 0]

a    1.970393
b    0.204419
c    0.939789
d    0.351338
e    0.577461
Name: A, dtype: float64

In [71]:
# Using these methods / indexers, you can chain data selection operations without using a temporary variable.
bb = pd.read_csv('data/baseball.csv', index_col='id')
(bb.groupby(['year', 'team']).sum(numeric_only=True)
   .loc[lambda df: df['r'] > 100])


stint    g    ab    r    h  X2b  X3b  hr    rbi    sb   cs   bb  \
year team                                                                    
2007 CIN       6  379   745  101  203   35    2  36  125.0  10.0  1.0  105   
     DET       5  301  1062  162  283   54    4  37  144.0  24.0  7.0   97   
     HOU       4  311   926  109  218   47    6  14   77.0  10.0  4.0   60   
     LAN      11  413  1021  153  293   61    3  36  154.0   7.0  5.0  114   
     NYN      13  622  1854  240  509  101    3  61  243.0  22.0  4.0  174   
     SFN       5  482  1305  198  337   67    6  40  171.0  26.0  7.0  235   
     TEX       2  198   729  115  200   40    4  28  115.0  21.0  4.0   73   
     TOR       4  459  1408  187  378   96    2  58  223.0   4.0  2.0  190   

              so   ibb   hbp    sh    sf  gidp  
year team                                       
2007 CIN   127.0  14.0   1.0   1.0  15.0  18.0  
     DET   176.0   3.0  10.0   4.0   8.0  28.0  
     HOU   212.0   3.0   9.0  16.0   6.0  17.0  
     LAN   141.0   8.0   9.0   3.0   8.0  29.0  
     NYN   310.0  24.0  23.0  18.0  15.0  48.0  
     SFN   188.0  51.0   8.0  16.0   6.0  41.0  
     TEX   140.0   4.0   5.0   2.0   8.0  16.0  
     TOR   265.0  16.0  12.0   4.0  16.0  38.0

### Combining positional and label-based indexing

In [72]:
#If you wish to get the 0th and the 2nd elements from the index in the ‘A’ column, you can do:
dfd = pd.DataFrame({'A': [1, 2, 3],
                    'B': [4, 5, 6]},
                   index=list('abc'))
dfd

,A,B
a,1,4
b,2,5
c,3,6


In [73]:
dfd.loc[dfd.index[[0, 2]], 'A']

a    1
c    3
Name: A, dtype: int64

In [74]:
# This can also be expressed using .iloc, by explicitly getting locations on the indexers, and using positional indexing to select things.
dfd.iloc[[0,2], dfd.columns.get_loc('A')]

a    1
c    3
Name: A, dtype: int64

In [75]:
# For getting multiple indexers, using .get_indexer:
dfd.iloc[[0, 2], dfd.columns.get_indexer(['A', 'B'])]

,A,B
a,1,4
c,3,6


### Reindexing
The idiomatic way to achieve selecting potentially not-found elements is via `.reindex()`. 

In [76]:
s = pd.Series([1, 2, 3])
s.reindex([1, 2, 3])

1    2.0
2    3.0
3    NaN
dtype: float64

In [77]:
# Alternatively, if you want to select only valid keys, the following is idiomatic and efficient; it is guaranteed to preserve the dtype of the selection.
labels = [1, 2, 3]
s.loc[s.index.intersection(labels)]

1    2
2    3
dtype: int64

In [78]:
# Having a duplicated index will raise for a .reindex():
s = pd.Series(np.arange(4), index=['a', 'a', 'b', 'c'])
labels = ['c', 'd']
s.reindex(labels)

ValueError: cannot reindex on an axis with duplicate labels

### Boolean indexing
Another common operation is the use of boolean vectors to filter the data. The operators are: `|` for `or`, `&` for `and`, and `~` for `not`. These **must** be grouped by using parentheses, since by default Python will evaluate an expression such as `df['A'] > 2 & df['B'] < 3` as `df['A'] > (2 & df['B']) < 3`, while the desired evaluation order is `(df['A'] > 2) & (df['B'] < 3)`.

Using a boolean vector to index a Series works exactly as in a NumPy ndarray:


In [79]:
s = pd.Series(range(-3, 4))
s

0   -3
1   -2
2   -1
3    0
4    1
5    2
6    3
dtype: int64

In [80]:
s[s>0]

4    1
5    2
6    3
dtype: int64

In [81]:
s[(s<-1) | (s > 0.5)] # and

0   -3
1   -2
4    1
5    2
6    3
dtype: int64

In [82]:
s[~(s < 0)] # not

3    0
4    1
5    2
6    3
dtype: int64

You may select rows from a DataFrame using a boolean vector the same length as the DataFrame’s index (for example, something derived from one of the columns of the DataFrame):

In [83]:
df

,A,B,C,D
2000-01-01,0.381918,-0.178866,-0.280091,-2.173926
2000-01-02,-2.165132,0.095747,0.599248,0.348885
2000-01-03,1.412580,0.028573,-0.702325,0.981729
2000-01-04,-0.492163,-0.565135,-0.502933,1.999355
2000-01-05,-0.293873,0.697005,0.694528,0.413838
2000-01-06,0.821495,0.085911,-0.790345,-0.040348
2000-01-07,0.326280,-1.481917,-0.470283,-0.741031
2000-01-08,-1.550474,0.015126,2.158909,3.557245


In [84]:
df[df['A'] > 0]

,A,B,C,D
2000-01-01,0.381918,-0.178866,-0.280091,-2.173926
2000-01-03,1.412580,0.028573,-0.702325,0.981729
2000-01-06,0.821495,0.085911,-0.790345,-0.040348
2000-01-07,0.326280,-1.481917,-0.470283,-0.741031


In [85]:
# List comprehensions and the map method of Series can also be used to produce more complex criteria:
df2 = pd.DataFrame({'a': ['one', 'one', 'two', 'three', 'two', 'one', 'six'],
                    'b': ['x', 'y', 'y', 'x', 'y', 'x', 'x'],
                    'c': np.random.randn(7)})
df2

,a,b,c
0,one,x,1.259906
1,one,y,0.853288
2,two,y,-0.614182
3,three,x,-1.017664
4,two,y,-0.237246
5,one,x,-0.023073
6,six,x,1.629206


In [86]:
# only want 'two' or 'three'
criterion = df2['a'].map(lambda x: x.startswith('t'))

In [87]:
df2[criterion]

,a,b,c
2,two,y,-0.614182
3,three,x,-1.017664
4,two,y,-0.237246


In [88]:
# equivalent but slower
df2[[x.startswith('t') for x in df2['a']]]

,a,b,c
2,two,y,-0.614182
3,three,x,-1.017664
4,two,y,-0.237246


In [89]:
# Multiple Criteria
df2[criterion & (df2['b'] == 'x')]

,a,b,c
3,three,x,-1.017664


In [90]:
# With the choice methods Selection by Label, Selection by Position, and Advanced Indexing you may select along more than one axis using boolean vectors combined with other indexing expressions.
df2.loc[criterion & (df2['b'] == 'x'), 'b':'c']

,b,c
3,x,-1.017664


##### ⚠️ Warning
`iloc` accepts a **boolean array**, but not a **boolean Series**.


While `loc` supports two kinds of boolean indexing, `iloc` only supports indexing with a boolean array. If the indexer is a boolean `Series`, an error will be raised. For instance, in the following example, `df.iloc[s.values, 1]` is ok. The boolean indexer is an array. But `df.iloc[s, 1]` would raise `ValueError`.


In [91]:
df = pd.DataFrame([[1, 2], [3, 4], [5, 6]],
                  index=list('abc'),
                  columns=['A', 'B'])
df

,A,B
a,1,2
b,3,4
c,5,6


In [92]:
s = (df['A'] > 2)
s

a    False
b     True
c     True
Name: A, dtype: bool

In [93]:
df.loc[s, 'B']

b    4
c    6
Name: B, dtype: int64

In [94]:
df.iloc[s.values, 1]

b    4
c    6
Name: B, dtype: int64

### Indexing with isin

Consider the **`isin()`** method of `Series`, which returns a boolean vector that is true wherever the `Series` elements exist in the passed list. This allows you to select rows where one or more columns have values you want:


In [95]:
s = pd.Series(np.arange(5), index=np.arange(5)[::-1], dtype='int64')
s

4    0
3    1
2    2
1    3
0    4
dtype: int64

In [96]:
s.isin([2, 4, 6])  # For every VALUE in s, check whether that value exists in [2, 4, 6].

4    False
3    False
2     True
1    False
0     True
dtype: bool

In [97]:
s[s.isin([2, 4, 6])]

2    2
0    4
dtype: int64

In [98]:
# The same method is available for Index objects and is useful for the cases when you don’t know which of the sought labels are in fact present:
s[s.index.isin([2, 4, 6])]

4    0
2    2
dtype: int64

In [99]:
# compare it to the following
s.reindex([2, 4, 6])

2    2.0
4    0.0
6    NaN
dtype: float64

In [100]:
# In addition to that, MultiIndex allows selecting a separate level to use in the membership check:
s_mi = pd.Series(np.arange(6),
                 index=pd.MultiIndex.from_product([[0, 1], ['a', 'b', 'c']]))
s_mi

0  a    0
   b    1
   c    2
1  a    3
   b    4
   c    5
dtype: int32

In [101]:
s_mi.iloc[s_mi.index.isin([(1, 'a'), (2, 'b'), (0, 'c')])]

0  c    2
1  a    3
dtype: int32

In [102]:
s_mi.iloc[s_mi.index.isin(['a', 'c', 'e'], level=1)]

0  a    0
   c    2
1  a    3
   c    5
dtype: int32

DataFrame also has an **`isin()`** method. When calling `isin`, pass a set of values as either an array or dict. If values is an array, `isin` returns a DataFrame of booleans that is the same shape as the original DataFrame, with `True` wherever the element is in the sequence of values.


In [103]:
df = pd.DataFrame({'vals': [1, 2, 3, 4], 'ids': ['a', 'b', 'f', 'n'],
                   'ids2': ['a', 'n', 'c', 'n']})
df


,vals,ids,ids2
0,1,a,a
1,2,b,n
2,3,f,c
3,4,n,n


In [104]:
values = ['a', 'b', 1, 3]
df.isin(values)

,vals,ids,ids2
0,True,True,True
1,False,True,False
2,True,False,False
3,False,False,False


In [105]:
# Oftentimes you’ll want to match certain values with certain columns. Just make values a dict where the key is the column, and the value is a list of items you want to check for.
values = {'ids': ['a', 'b'], 'vals': [1, 3]}
df.isin(values)

,vals,ids,ids2
0,True,True,False
1,False,True,False
2,True,False,False
3,False,False,False


In [106]:
# To return the DataFrame of booleans where the values are not in the original DataFrame, use the ~ operator:
values = {'ids': ['a', 'b'], 'vals': [1, 3]}

~df.isin(values)

,vals,ids,ids2
0,False,False,True
1,True,False,True
2,False,True,True
3,True,True,True


In [107]:
# Combine DataFrame’s isin with the any() and all() methods to quickly select subsets of your data that meet a given criteria. To select a row where each column meets its own criterion:
values = {'ids': ['a', 'b'], 'ids2': ['a', 'c'], 'vals': [1, 3]}

row_mask = df.isin(values).all(axis=1)

df[row_mask]

,vals,ids,ids2
0,1,a,a


### The where() Method and Masking

Selecting values from a Series with a boolean vector generally returns a subset of the data. To guarantee that selection output has the same shape as the original data, you can use the `where` method in `Series` and `DataFrame`.


In [110]:
# to return only the selected rows:
s[s>0]

3    1
2    2
1    3
0    4
dtype: int64

In [116]:
# To return a Series of the same shape as the original:
s.where(s > 0)

4    NaN
3    1.0
2    2.0
1    3.0
0    4.0
dtype: float64

Selecting values from a DataFrame with a boolean criterion now also preserves input data shape. where is used under the hood as the implementation. The code below is equivalent to `df.where(df < 0)`.

In [120]:
dates = pd.date_range('1/1/2000', periods = 8)
df = pd.DataFrame(np.random.randn(8, 4),
                 index = dates, columns = ['A', 'B', 'C', 'D'])
df

,A,B,C,D
2000-01-01,-0.361685,-0.519544,1.813564,2.465817
2000-01-02,0.968803,0.239125,0.300433,-1.455825
2000-01-03,0.792118,-0.657164,0.291520,0.930473
2000-01-04,0.353096,0.472979,-0.281137,0.759699
2000-01-05,-0.582783,0.099605,-1.167010,0.535747
2000-01-06,-0.859659,0.170106,-0.358804,0.676483
2000-01-07,0.158077,0.019399,0.897343,0.884483
2000-01-08,0.073040,0.414915,-1.272156,-0.033340


In [121]:
df[df < 0]

,A,B,C,D
2000-01-01,-0.361685,-0.519544,NaN,NaN
2000-01-02,NaN,NaN,NaN,-1.455825
2000-01-03,NaN,-0.657164,NaN,NaN
2000-01-04,NaN,NaN,-0.281137,NaN
2000-01-05,-0.582783,NaN,-1.167010,NaN
2000-01-06,-0.859659,NaN,-0.358804,NaN
2000-01-07,NaN,NaN,NaN,NaN
2000-01-08,NaN,NaN,-1.272156,-0.033340


In [122]:
# In addition, where takes an optional other argument for replacement of values where the condition is False, in the returned copy.
df.where(df < 0, -df)

,A,B,C,D
2000-01-01,-0.361685,-0.519544,-1.813564,-2.465817
2000-01-02,-0.968803,-0.239125,-0.300433,-1.455825
2000-01-03,-0.792118,-0.657164,-0.291520,-0.930473
2000-01-04,-0.353096,-0.472979,-0.281137,-0.759699
2000-01-05,-0.582783,-0.099605,-1.167010,-0.535747
2000-01-06,-0.859659,-0.170106,-0.358804,-0.676483
2000-01-07,-0.158077,-0.019399,-0.897343,-0.884483
2000-01-08,-0.073040,-0.414915,-1.272156,-0.033340


In [123]:
# You may wish to set values based on some boolean criteria. This can be done intuitively like so:
s2 = s.copy()
s2

4    0
3    1
2    2
1    3
0    4
dtype: int64

In [126]:
s2[s2< 0] = 0
s2

4    0
3    1
2    2
1    3
0    4
dtype: int64

In [127]:
df2 = df.copy()

In [128]:
df2[df2 < 0] = 0
df2

,A,B,C,D
2000-01-01,0.000000,0.000000,1.813564,2.465817
2000-01-02,0.968803,0.239125,0.300433,0.000000
2000-01-03,0.792118,0.000000,0.291520,0.930473
2000-01-04,0.353096,0.472979,0.000000,0.759699
2000-01-05,0.000000,0.099605,0.000000,0.535747
2000-01-06,0.000000,0.170106,0.000000,0.676483
2000-01-07,0.158077,0.019399,0.897343,0.884483
2000-01-08,0.073040,0.414915,0.000000,0.000000


**Alignment**

Furthermore, `where` aligns the input boolean condition (ndarray or DataFrame), such that partial selection with setting is possible. This is analogous to partial setting via `.loc` (but on the contents rather than the axis labels).


In [129]:
df2 = df.copy()

df2[df2[1:4] > 0] = 3

df2

,A,B,C,D
2000-01-01,-0.361685,-0.519544,1.813564,2.465817
2000-01-02,3.000000,3.000000,3.000000,-1.455825
2000-01-03,3.000000,-0.657164,3.000000,3.000000
2000-01-04,3.000000,3.000000,-0.281137,3.000000
2000-01-05,-0.582783,0.099605,-1.167010,0.535747
2000-01-06,-0.859659,0.170106,-0.358804,0.676483
2000-01-07,0.158077,0.019399,0.897343,0.884483
2000-01-08,0.073040,0.414915,-1.272156,-0.033340


In [131]:
# Where can also accept axis and level parameters to align the input when performing the where.
df2 = df.copy()
df2.where(df2 > 0, df['A'], axis = 'index')

,A,B,C,D
2000-01-01,-0.361685,-0.361685,1.813564,2.465817
2000-01-02,0.968803,0.239125,0.300433,0.968803
2000-01-03,0.792118,0.792118,0.291520,0.930473
2000-01-04,0.353096,0.472979,0.353096,0.759699
2000-01-05,-0.582783,0.099605,-0.582783,0.535747
2000-01-06,-0.859659,0.170106,-0.859659,0.676483
2000-01-07,0.158077,0.019399,0.897343,0.884483
2000-01-08,0.073040,0.414915,0.073040,0.073040


In [133]:
# This is equivalent to (but faster than) the following
df2 = df.copy()
df.apply(lambda x, y: x.where(x > 0, y), y = df['A'])

,A,B,C,D
2000-01-01,-0.361685,-0.361685,1.813564,2.465817
2000-01-02,0.968803,0.239125,0.300433,0.968803
2000-01-03,0.792118,0.792118,0.291520,0.930473
2000-01-04,0.353096,0.472979,0.353096,0.759699
2000-01-05,-0.582783,0.099605,-0.582783,0.535747
2000-01-06,-0.859659,0.170106,-0.859659,0.676483
2000-01-07,0.158077,0.019399,0.897343,0.884483
2000-01-08,0.073040,0.414915,0.073040,0.073040


In [134]:
# where can accept a callable as condition and other arguments. The function must be with one argument (the calling Series or DataFrame) and that returns valid output as condition and other argument.
df3 = pd.DataFrame({'A': [1, 2, 3],
                    'B': [4, 5, 6],
                    'C': [7, 8, 9]})


df3.where(lambda x: x > 4, lambda x: x + 10)

,A,B,C
0,11,14,7
1,12,5,8
2,13,6,9


### Mask

`mask()` is the inverse boolean operation of where.

In [135]:
s.mask(s >= 0)

4   NaN
3   NaN
2   NaN
1   NaN
0   NaN
dtype: float64

In [136]:
df.mask(df>= 0)

,A,B,C,D
2000-01-01,-0.361685,-0.519544,NaN,NaN
2000-01-02,NaN,NaN,NaN,-1.455825
2000-01-03,NaN,-0.657164,NaN,NaN
2000-01-04,NaN,NaN,-0.281137,NaN
2000-01-05,-0.582783,NaN,-1.167010,NaN
2000-01-06,-0.859659,NaN,-0.358804,NaN
2000-01-07,NaN,NaN,NaN,NaN
2000-01-08,NaN,NaN,-1.272156,-0.033340


### The Query() Method

`DataFrame` objects have a `query()` method that allows selection using an expression.

You can get the value of the frame where column b has values between the values of columns a and c. For example:

In [137]:
n = 10
df = pd.DataFrame(np.random.rand(n, 3), columns = list('abc'))
df

,a,b,c
0,0.683953,0.641583,0.910689
1,0.167903,0.392124,0.175003
2,0.243156,0.876106,0.325574
3,0.493927,0.998161,0.008906
4,0.566259,0.088617,0.093909
5,0.560628,0.990479,0.384777
6,0.069441,0.926470,0.303354
7,0.906757,0.482816,0.094936
8,0.149538,0.196905,0.628638
9,0.234708,0.713633,0.085682


In [138]:
# pure python
df[(df['a']< df['b']) & (df['b'] < df['c'])]

,a,b,c
8,0.149538,0.196905,0.628638


In [139]:
# query
df.query('(a < b) & (b < c)')

,a,b,c
8,0.149538,0.196905,0.628638


In [146]:
# Do the same thing but fall back on a named index if there is no column with the name a.
df = pd.DataFrame(np.random.randint(n / 2, size=(n, 2)), columns=list('bc'))
df.index.name = 'a'
df

,b,c
a,,
0,2,4
1,2,4
2,1,4
3,1,4
4,1,1
5,2,2
6,3,1
7,3,3
8,0,1


In [147]:
df.query('a<b and b < c')

,b,c
a,,
0,2,4
1,2,4


In [148]:
# If instead you don’t want to or cannot name your index, you can use the name index in your query expression:
df = pd.DataFrame(np.random.randint(n, size=(n, 2)), columns=list('bc'))

df

,b,c
0,7,8
1,4,8
2,0,0
3,3,8
4,8,4
5,5,2
6,8,6
7,2,3
8,4,2
9,2,6


In [149]:
df.query('index < b < c')

,b,c
0,7,8
1,4,8


In [150]:
# You can still use the index in a query expression by using the special identifier ‘index’:
df.query('index > 2')

,b,c
3,3,8
4,8,4
5,5,2
6,8,6
7,2,3
8,4,2
9,2,6
